# Monthly NDVI Mosaics from Phase 2 Filtered Data

**Project:** 8-Band Satellite Imagery Processing for Ghost River Wilderness Area

**Purpose:** Create monthly NDVI mosaics by averaging bi-weekly periods

**Output Structure:**
```
S:/8band_processing_workflow/monthly_mosaics/
├── 2022/
│   ├── NDVI_2022_May.tif
│   ├── NDVI_2022_June.tif
│   ├── NDVI_2022_July.tif
│   ├── NDVI_2022_August.tif
│   ├── NDVI_2022_September.tif
│   └── NDVI_2022_October.tif
├── 2023/
│   └── ... (same structure)
├── 2024/
│   └── ... (same structure)
└── all_years_avg/
    ├── NDVI_allyears_May.tif
    ├── NDVI_allyears_June.tif
    ├── NDVI_allyears_July.tif
    ├── NDVI_allyears_August.tif
    ├── NDVI_allyears_September.tif
    └── NDVI_allyears_October.tif
```

**Workflow:**
1. Setup and configuration
2. Define common spatial grid (like Phase 3)
3. Create monthly mosaics for each year (2022, 2023, 2024)
4. Create cross-year monthly averages

---

## 1. Setup and Configuration

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds
import geopandas as gpd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded successfully")
print(f"  Rasterio: {rasterio.__version__}")
print(f"  NumPy: {np.__version__}")
print(f"  GeoPandas: {gpd.__version__}")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# File paths
BASE_DIR = Path('S:/8band_processing_workflow')
FILTERED_DIR = BASE_DIR / 'ndvi_filtered'
OUTPUT_DIR = BASE_DIR / 'monthly_mosaics'

# Years to process
YEARS = [2022, 2023, 2024]

# Landcover shapefiles (for defining common grid)
LANDCOVER_PATHS = {
    2022: Path('<PROJECT_ROOT>/gis_layers/strata_2022/herb_shrub_wetland_2022.shp'),
    2023: Path('<PROJECT_ROOT>/gis_layers/strata_2023/herb_shrub_wetland_2023.shp'),
    2024: Path('<PROJECT_ROOT>/gis_layers/strata_2024/herb_shrub_wetland_2024.shp')
}

# Spatial parameters
TARGET_CRS = 'EPSG:3400'  # NAD 1983 10TM AEP Forest
PIXEL_SIZE = 3.0  # meters

# Months to process
MONTHS = ['May', 'June', 'July', 'August', 'September', 'October']

# Create output directories
for year in YEARS:
    (OUTPUT_DIR / str(year)).mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'all_years_avg').mkdir(parents=True, exist_ok=True)

print("\n✓ Configuration complete")
print(f"  Input: {FILTERED_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Target CRS: {TARGET_CRS}")
print(f"  Pixel size: {PIXEL_SIZE}m")
print(f"  Years: {YEARS}")
print(f"  Months: {MONTHS}")

## 2. Define Period Mappings

In [ ]:
# ============================================================================
# BIWEEKLY PERIOD DEFINITIONS
# ============================================================================

# All possible bi-weekly periods
BIWEEKLY_PERIODS = [
    'May01-15', 'May16-31',
    'Jun01-15', 'Jun16-30',
    'Jul01-15', 'Jul16-31',
    'Aug01-15', 'Aug16-31',
    'Sep01-15', 'Sep16-30',
    'Oct01-15'
]

def period_to_month(period_name):
    """
    Map bi-weekly period to month name.
    
    Parameters
    ----------
    period_name : str
        Period name (e.g., 'May01-15', 'Jun16-30')
        
    Returns
    -------
    str or None
        Month name or None if not recognized
    """
    if period_name.startswith('May'):
        return 'May'
    elif period_name.startswith('Jun'):
        return 'June'
    elif period_name.startswith('Jul'):
        return 'July'
    elif period_name.startswith('Aug'):
        return 'August'
    elif period_name.startswith('Sep'):
        return 'September'
    elif period_name.startswith('Oct'):
        return 'October'
    return None

# Test mapping
print("Period to Month mapping:")
for period in BIWEEKLY_PERIODS:
    month = period_to_month(period)
    print(f"  {period:15s} -> {month}")

## 3. Load Landcover and Define Common Grid

In [ ]:
print("\n" + "="*80)
print("LOADING LANDCOVER AND DEFINING COMMON GRID")
print("="*80)

# Load all years' landcover
print("\nLoading landcover for all years...")
landcover_gdfs = {}
for year, path in LANDCOVER_PATHS.items():
    print(f"  {year}: {path.name}")
    gdf = gpd.read_file(path)
    if gdf.crs != TARGET_CRS:
        print(f"    Reprojecting to {TARGET_CRS}")
        gdf = gdf.to_crs(TARGET_CRS)
    landcover_gdfs[year] = gdf

# Combine all years to get maximum extent
all_landcover = pd.concat([gdf for gdf in landcover_gdfs.values()], ignore_index=True)
all_landcover_gdf = gpd.GeoDataFrame(all_landcover, crs=TARGET_CRS)

print(f"\nCombined landcover:")
print(f"  Total polygons: {len(all_landcover_gdf)}")
print(f"  Total area: {all_landcover_gdf.area.sum() / 1e6:.2f} km²")

# Get bounding box from combined landcover
bounds = all_landcover_gdf.total_bounds  # [minx, miny, maxx, maxy]
minx, miny, maxx, maxy = bounds

print(f"\nCombined extent:")
print(f"  Bounds: {bounds}")

In [ ]:
# ============================================================================
# DEFINE COMMON REFERENCE GRID
# ============================================================================

# Calculate dimensions
width = int(np.ceil((maxx - minx) / PIXEL_SIZE))
height = int(np.ceil((maxy - miny) / PIXEL_SIZE))

# Adjust max coordinates to match pixel grid
maxx_adjusted = minx + (width * PIXEL_SIZE)
maxy_adjusted = miny + (height * PIXEL_SIZE)

# Create reference transform
reference_transform = from_bounds(minx, miny, maxx_adjusted, maxy_adjusted, width, height)

print("\n✓ Reference grid defined:")
print(f"  Dimensions: {height} × {width} = {height * width:,} pixels")
print(f"  Pixel size: {PIXEL_SIZE}m × {PIXEL_SIZE}m")
print(f"  Extent: {minx:.1f}, {miny:.1f}, {maxx_adjusted:.1f}, {maxy_adjusted:.1f}")
print(f"  Transform: {reference_transform}")

# Store for use in subsequent cells
reference_shape = (height, width)
reference_crs = TARGET_CRS

print(f"\n✓ Variables stored for downstream use:")
print(f"  reference_shape: {reference_shape}")
print(f"  reference_crs: {reference_crs}")

## 4. Helper Function: Load to Common Grid

In [ ]:
def load_to_common_grid(src_path, reference_transform, reference_shape, reference_crs):
    """
    Load raster to common grid.
    
    Always reprojects to ensure perfect alignment across all files.
    
    Parameters
    ----------
    src_path : Path
        Path to source raster
    reference_transform : Affine
        Target transform
    reference_shape : tuple
        Target shape (height, width)
    reference_crs : CRS
        Target CRS
        
    Returns
    -------
    data : ndarray
        Reprojected data on common grid
    """
    with rasterio.open(src_path) as src:
        # Check if already aligned
        if (src.shape == reference_shape and 
            str(src.crs) == str(reference_crs) and
            abs(src.transform.a - reference_transform.a) < 0.01 and
            abs(src.transform.e - reference_transform.e) < 0.01 and
            abs(src.transform.c - reference_transform.c) < 1.0 and
            abs(src.transform.f - reference_transform.f) < 1.0):
            # Already aligned - read directly
            data = src.read(1, masked=True).filled(np.nan).astype(np.float32)
        else:
            # Need to reproject
            data = np.full(reference_shape, np.nan, dtype=np.float32)
            
            reproject(
                source=rasterio.band(src, 1),
                destination=data,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=reference_transform,
                dst_crs=reference_crs,
                resampling=Resampling.bilinear,
                src_nodata=src.nodata,
                dst_nodata=np.nan
            )
        
        return data

print("✓ Loading function defined")

## 5. Inventory Available Files

In [ ]:
print("\n" + "="*80)
print("INVENTORYING PHASE 2 FILTERED FILES")
print("="*80)

# Find all files for all years
all_year_files = {}

for year in YEARS:
    year_dir = FILTERED_DIR / str(year)
    
    if not year_dir.exists():
        print(f"\n⚠ WARNING: Directory not found for {year}: {year_dir}")
        all_year_files[year] = {}
        continue
    
    # Find all mosaic files
    files = sorted(year_dir.glob(f"mosaic_{year}_*.tif"))
    
    print(f"\n{year}: Found {len(files)} files")
    
    # Group by month
    monthly_files = {month: [] for month in MONTHS}
    
    for filepath in files:
        # Extract period from filename: mosaic_YYYY_Period.tif
        # Example: mosaic_2024_May01-15.tif -> May01-15
        parts = filepath.name.replace('.tif', '').split('_')
        
        if len(parts) >= 3:
            period_name = '_'.join(parts[2:])
            month = period_to_month(period_name)
            
            if month and month in monthly_files:
                monthly_files[month].append(filepath)
                print(f"  {filepath.name:45s} -> {month}")
    
    all_year_files[year] = monthly_files
    
    # Summary for this year
    print(f"\n  Summary for {year}:")
    for month in MONTHS:
        count = len(monthly_files[month])
        if count > 0:
            print(f"    {month:10s}: {count} periods")
        else:
            print(f"    {month:10s}: {count} periods (⚠ NO DATA)")

print("\n" + "="*80)

## 6. Create Monthly Mosaics for Each Year

In [ ]:
print("\n" + "="*80)
print("CREATING MONTHLY NDVI MOSAICS")
print("="*80)

# Store all monthly mosaics for cross-year averaging
all_monthly_mosaics = {month: {} for month in MONTHS}  # month -> {year: array}

for year in YEARS:
    print(f"\n{'='*80}")
    print(f"PROCESSING YEAR: {year}")
    print(f"{'='*80}")
    
    monthly_files = all_year_files[year]
    
    for month in MONTHS:
        files = monthly_files[month]
        
        if len(files) == 0:
            print(f"\n{month} ({year}): ⚠ NO DATA - skipping")
            continue
        
        print(f"\n{month} ({year}): {len(files)} periods")
        print("-" * 80)
        
        # Load all periods for this month
        period_arrays = []
        
        for filepath in files:
            print(f"  Loading {filepath.name}...", end=" ")
            
            data = load_to_common_grid(
                filepath,
                reference_transform,
                reference_shape,
                reference_crs
            )
            
            valid = np.sum(~np.isnan(data))
            print(f"{valid:,} valid pixels")
            
            period_arrays.append(data)
        
        # Calculate mean across periods (using nanmean to handle missing data)
        print(f"  Calculating monthly average...")
        monthly_stack = np.array(period_arrays)
        
        # Use nanmean along axis 0 (across periods)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            monthly_composite = np.nanmean(monthly_stack, axis=0).astype(np.float32)
        
        # Calculate statistics
        valid_pixels = np.sum(~np.isnan(monthly_composite))
        total_pixels = reference_shape[0] * reference_shape[1]
        coverage = valid_pixels / total_pixels * 100
        
        valid_values = monthly_composite[~np.isnan(monthly_composite)]
        if len(valid_values) > 0:
            mean_ndvi = np.mean(valid_values)
            std_ndvi = np.std(valid_values)
            min_ndvi = np.min(valid_values)
            max_ndvi = np.max(valid_values)
            
            print(f"  Composite: {valid_pixels:,} pixels ({coverage:.2f}% coverage)")
            print(f"  NDVI: {mean_ndvi:.3f} ± {std_ndvi:.3f} (range: {min_ndvi:.3f} - {max_ndvi:.3f})")
        else:
            print(f"  ⚠ WARNING: No valid pixels in composite!")
        
        # Store for cross-year averaging
        all_monthly_mosaics[month][year] = monthly_composite
        
        # Save to file
        output_file = OUTPUT_DIR / str(year) / f"NDVI_{year}_{month}.tif"
        
        profile = {
            'driver': 'GTiff',
            'dtype': 'float32',
            'nodata': np.nan,
            'width': reference_shape[1],
            'height': reference_shape[0],
            'count': 1,
            'crs': reference_crs,
            'transform': reference_transform,
            'compress': 'lzw',
            'tiled': True,
            'blockxsize': 256,
            'blockysize': 256
        }
        
        with rasterio.open(output_file, 'w', **profile) as dst:
            dst.write(monthly_composite, 1)
        
        print(f"  ✓ Saved: {output_file.name}")

print(f"\n{'='*80}")
print("✓ All yearly monthly mosaics complete")
print(f"{'='*80}")

## 7. Create Cross-Year Monthly Averages

In [ ]:
print("\n" + "="*80)
print("CREATING CROSS-YEAR MONTHLY AVERAGES")
print("="*80)

for month in MONTHS:
    print(f"\n{month}:")
    print("-" * 80)
    
    # Collect all years that have data for this month
    year_arrays = []
    available_years = []
    
    for year in YEARS:
        if year in all_monthly_mosaics[month]:
            year_arrays.append(all_monthly_mosaics[month][year])
            available_years.append(year)
            valid = np.sum(~np.isnan(all_monthly_mosaics[month][year]))
            print(f"  {year}: {valid:,} valid pixels")
    
    if len(year_arrays) == 0:
        print(f"  ⚠ NO DATA for {month} across any year - skipping")
        continue
    
    print(f"  Averaging across {len(year_arrays)} years: {available_years}")
    
    # Stack and calculate mean
    year_stack = np.array(year_arrays)
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        cross_year_avg = np.nanmean(year_stack, axis=0).astype(np.float32)
    
    # Calculate statistics
    valid_pixels = np.sum(~np.isnan(cross_year_avg))
    total_pixels = reference_shape[0] * reference_shape[1]
    coverage = valid_pixels / total_pixels * 100
    
    valid_values = cross_year_avg[~np.isnan(cross_year_avg)]
    if len(valid_values) > 0:
        mean_ndvi = np.mean(valid_values)
        std_ndvi = np.std(valid_values)
        min_ndvi = np.min(valid_values)
        max_ndvi = np.max(valid_values)
        
        print(f"  Cross-year average: {valid_pixels:,} pixels ({coverage:.2f}% coverage)")
        print(f"  NDVI: {mean_ndvi:.3f} ± {std_ndvi:.3f} (range: {min_ndvi:.3f} - {max_ndvi:.3f})")
    
    # Save to file
    output_file = OUTPUT_DIR / 'all_years_avg' / f"NDVI_allyears_{month}.tif"
    
    profile = {
        'driver': 'GTiff',
        'dtype': 'float32',
        'nodata': np.nan,
        'width': reference_shape[1],
        'height': reference_shape[0],
        'count': 1,
        'crs': reference_crs,
        'transform': reference_transform,
        'compress': 'lzw',
        'tiled': True,
        'blockxsize': 256,
        'blockysize': 256
    }
    
    with rasterio.open(output_file, 'w', **profile) as dst:
        dst.write(cross_year_avg, 1)
    
    print(f"  ✓ Saved: {output_file.name}")

print(f"\n{'='*80}")
print("✓ All cross-year monthly averages complete")
print(f"{'='*80}")

## 8. Final Summary

In [ ]:
print("\n" + "="*80)
print("MONTHLY NDVI MOSAICS - COMPLETE SUMMARY")
print("="*80)

print("\nOutput structure created:")
print(f"  {OUTPUT_DIR}")

for year in YEARS:
    year_dir = OUTPUT_DIR / str(year)
    files = sorted(year_dir.glob('*.tif'))
    print(f"\n  {year}/")
    for f in files:
        print(f"    {f.name}")

allyears_dir = OUTPUT_DIR / 'all_years_avg'
files = sorted(allyears_dir.glob('*.tif'))
print(f"\n  all_years_avg/")
for f in files:
    print(f"    {f.name}")

print("\n" + "="*80)
print("✓ WORKFLOW COMPLETE")
print("="*80)
print("\nNext steps:")
print("  1. Verify spatial alignment of output mosaics")
print("  2. Visual QC of monthly patterns")
print("  3. Compare monthly averages across years")
print("  4. Use in habitat modeling or biomass correlation analysis")